# Pydantic model to structure output

TODO: create an agent to simulate IT employees

In [15]:
from pydantic_ai import Agent
from dotenv import load_dotenv

load_dotenv()

agent = Agent(
    "openrouter:liquid/lfm-2.5-1.2b-instruct:free",
    system_prompt="You are an IT expert that answers basic IT questiones that would come to an IT desk. Come up with several solutions to the users' problem(s).")

prompt = "I am locked out of my laptop"

result = await agent.run(prompt)

result

AgentRunResult(output="Hello! I'm sorry to hear you're locked out of your laptop. There are several potential solutions depending on the circumstances. Here are some common possibilities and how you can try to resolve the issue:\n\n### 1. **Change the Password**\n   - **What to do:** Try the most recent password you remember or re-enter the last time you managed to access the laptop securely.\n   - **Tip:** If the password is too weak, consider using a password manager or generating a new one.\n\n### 2. **Reset Password Remotely**\n   - If you have another computer or access to the device remotely, you can reset the password using settings or cloud services.\n   - **Steps:**\n     - Check your email or cloud storage for a reset link or password reset instructions.\n     - Use your phone or a computer to reset the password on the other device.\n\n### 3. **Enable Remote Support**\n   - If you have a working phone or computer nearby, enable **remote access** or **remote support** through 

In [16]:
print(result.output)

Hello! I'm sorry to hear you're locked out of your laptop. There are several potential solutions depending on the circumstances. Here are some common possibilities and how you can try to resolve the issue:

### 1. **Change the Password**
   - **What to do:** Try the most recent password you remember or re-enter the last time you managed to access the laptop securely.
   - **Tip:** If the password is too weak, consider using a password manager or generating a new one.

### 2. **Reset Password Remotely**
   - If you have another computer or access to the device remotely, you can reset the password using settings or cloud services.
   - **Steps:**
     - Check your email or cloud storage for a reset link or password reset instructions.
     - Use your phone or a computer to reset the password on the other device.

### 3. **Enable Remote Support**
   - If you have a working phone or computer nearby, enable **remote access** or **remote support** through your laptop provider or company IT d

In [ ]:
# with structured system prompt for better/structured output

employee_simulator_agent = Agent("openrouter:liquid/lfm-2.5-1.2b-thinking:free",
system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.

Fields to include in output:
- name
- age
- gender
- job_title
- salary in SEK per month
""")

result = await employee_simulator_agent.run("Simulate two employees")

print(result.output)

Here are two simulated employees in Sweden's data science and IT field:

---

**Employee 1**  
- **name**: EmmaHansson  
- **age**: 32  
- **gender**: Female  
- **job_title**: Data Analyst  
- **salary**: 480,000 SEK/month  

**Employee 2**  
- **name**: Jan Lundgren  
- **age**: 35  
- **gender**: Male  
- **job_title**: Machine Learning Engineer  
- **salary**: 520,000 SEK/month  

--- 

These profiles reflect typical roles in Sweden's IT/data science ecosystem. Let me know if you'd like further customization!


In [10]:
with open("simulated_employees.md", "w") as file:
    file.write(result.output)

## Get more structured output

Issues with above:
- output structure vary
- hard to work with the data e.g. compute mean of salaries

want:
- repeatable structure

In [ ]:
from pydantic_ai import Agent
from pydantic import BaseModel, Field
from typing import Literal

class EmployeeModel(BaseModel):
    name: str = Field(
        description="Mostly Swedish names, but could be foreign names as well")
    age: int = Field(
        ge=18,
        le=67,
        description="Age should be between 18 and 67")
    gender: Literal["Male", "Female"]
    experience_level: Literal["Entry", "Mid level", "Senior", "Expert"]
    job_title: str
    salary: int = Field(
        gt=30_000,
        lt=50_000,
        description="Salary should be between 30k and 50k. The higher the experience level, the higher the salary.")

employee_simulator_agent = Agent("openrouter:nvidia/nemotron-nano-12b-v2-vl:free",
system_prompt="""
You are an HR expert within IT field in Sweden within data science, data engineering,
machine learning, AI engineering. You will simulate IT employees.
""", retries=1)

result = await employee_simulator_agent.run("Give me 3 employees", output_type=EmployeeModel)
result

# we are only getting single employee - as fix we could type it as list so that the agent iterates

AgentRunResult(output=EmployeeModel(name='Anna Johansson', age=28, gender='Female', experience_level='Entry', job_title='Data Scientist', salary=35000))

In [25]:
result.output

EmployeeModel(name='Anna Johansson', age=28, gender='Female', experience_level='Entry', job_title='Data Scientist', salary=35000)

In [ ]:
# direct access to fields and with correct types so that we can manage them
result.output.salary

35000

In [27]:
result.output.salary+5000

40000

In [ ]:
# fix so that the model is itirated as mentioned in the user prompt (3 employees)
result = await employee_simulator_agent.run("Give me 3 employees", output_type=list[EmployeeModel])
result 

AgentRunResult(output=[EmployeeModel(name='Erik', age=32, gender='Male', experience_level='Mid level', job_title='Data Scientist', salary=38000), EmployeeModel(name='Anna', age=28, gender='Female', experience_level='Entry', job_title='Machine Learning Engineer', salary=33000), EmployeeModel(name='Lisa', age=39, gender='Female', experience_level='Senior', job_title='Data Engineer', salary=47000)])

In [ ]:
# BaseModel -> dictionary
result.output[0].model_dump()

{'name': 'Erik',
 'age': 32,
 'gender': 'Male',
 'experience_level': 'Mid level',
 'job_title': 'Data Scientist',
 'salary': 38000}

In [ ]:
result.output[1].name, result.output[1].salary

('Anna', 33000)

In [41]:
for employee in result.output:
    print(employee.name)

Erik
Anna
Lisa


In [42]:
for employee in result.output:
    print(employee.name, employee.salary)

Erik 38000
Anna 33000
Lisa 47000


In [40]:
name_salary = [(employee.name, employee.salary) for employee in result.output]
print(name_salary) 

[('Erik', 38000), ('Anna', 33000), ('Lisa', 47000)]


In [43]:
name_salary = [{"name": employee.name, "salary":employee.salary} for employee in result.output]
print(name_salary) 

[{'name': 'Erik', 'salary': 38000}, {'name': 'Anna', 'salary': 33000}, {'name': 'Lisa', 'salary': 47000}]


TODO:
- result.output make into list of dicts
- create pandas dataframe based on the list

In [59]:
for employee in result.output:
    print(employee.model_dump())

{'name': 'Erik', 'age': 32, 'gender': 'Male', 'experience_level': 'Mid level', 'job_title': 'Data Scientist', 'salary': 38000}
{'name': 'Anna', 'age': 28, 'gender': 'Female', 'experience_level': 'Entry', 'job_title': 'Machine Learning Engineer', 'salary': 33000}
{'name': 'Lisa', 'age': 39, 'gender': 'Female', 'experience_level': 'Senior', 'job_title': 'Data Engineer', 'salary': 47000}


In [64]:
list_employee = [employee.model_dump() for employee in result.output]
list_employee

[{'name': 'Erik',
  'age': 32,
  'gender': 'Male',
  'experience_level': 'Mid level',
  'job_title': 'Data Scientist',
  'salary': 38000},
 {'name': 'Anna',
  'age': 28,
  'gender': 'Female',
  'experience_level': 'Entry',
  'job_title': 'Machine Learning Engineer',
  'salary': 33000},
 {'name': 'Lisa',
  'age': 39,
  'gender': 'Female',
  'experience_level': 'Senior',
  'job_title': 'Data Engineer',
  'salary': 47000}]

In [67]:
import pandas as pd
df = pd.DataFrame(list_employee)
df

,name,age,gender,experience_level,job_title,salary
0,Erik,32,Male,Mid level,Data Scientist,38000
1,Anna,28,Female,Entry,Machine Learning Engineer,33000
2,Lisa,39,Female,Senior,Data Engineer,47000


In [68]:
df["salary"].mean()

np.float64(39333.333333333336)

In [71]:
df.to_csv("similated_employees.csv", index=False)